In [ ]:
import s3fs #To access cloud files as though they were local
from pprint import pprint
import asdf #To handle ASDF input and output
import roman_datamodels as rdm #To read Roman WFI ASDF files
from astrocut import get_center_pixel, ASDFCutout, FITSCutout #The main package import, used to generate the cutouts from the input file
import pyarrow.parquet as pq

from copy import deepcopy

# To manage the output FITS data file type and handle input sky coordinates
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS
from astropy.convolution import Gaussian2DKernel, convolve_fft
from astropy.wcs.utils import proj_plane_pixel_scales
from astropy.nddata import Cutout2D
from astropy.io import fits
from astropy.table import Table

from astroquery.sdss import SDSS # To query SDSS database for specific sources in our FOV
from astroquery.skyview import SkyView
from astroquery.utils.tap.core import TapPlus
# NOIRLab Data Lab TAP service
tap_url = "https://datalab.noirlab.edu/tap"
tap = TapPlus(url=tap_url)

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Rectangle

import numpy as np

from photutils.segmentation import detect_threshold, detect_sources, SourceCatalog

import os

import pandas as pd
import astrocut
print(rdm.__version__)
print(astrocut.__version__)

In [ ]:
def open_image_asdf(asdf_file_uri, display = False):
    """
    Open an asdf file image and return the 
    image array, WCS info, and pixel scale
    """
    with fs.open(asdf_file_uri, 'rb') as f: #opens the file in binary read mode; Python file object streaming from S3
        with asdf.open(f) as af: #Reads the file using the asdf library; is now an AsdfFile object containing structured scientific data (like a tree/dictionary)
            print(af)
            dm = rdm.open(af) #Converts the ASDF file into a higher-level data model object
            data = deepcopy(dm.data)  # Loads the WFI data into memory
            gwcs = deepcopy(dm.meta.wcs)  # Loads the WCS information into memory

    header_gwcs = gwcs.to_fits_sip()
    wcs = WCS(header_gwcs)
    pixel_scale = proj_plane_pixel_scales(wcs) * 3600  # arcsec/pixel
    print('The pixel scale is: ', pixel_scale)

    if display == True:
        print('Displaying original image read from file...')    
        plt.figure(figsize=(10, 8))
        plt.imshow(data, vmin=0, vmax=5, origin='lower')
        plt.show()


    return data, [wcs,gwcs], pixel_scale

def create_image_cutout_test(asdf_file_uri, cutout_size, position, display_cut = False, display_full=False, full_data = [None,None]):
    """
    Creates an image cutout using ASDF cutout (imcut)
    """


    ra, dec = position[0],position[1]
    center_coord = SkyCoord(ra, dec, unit='deg')
    
    
    
    
    if display_full == True:
        r_x, r_y = wcs_.wcs_world2pix(ra_, dec_, 1)
        data, wcs = full_data[0], full_data[1]
        rect = Rectangle((r_x-(cutout_size[0]/2), r_y-(cutout_size[1]/2)), (cutout_size[0]), (cutout_size[1]), linewidth=1, edgecolor='r', facecolor='none')
        plt.figure(figsize=(10, 8))
        ax2 = plt.subplot(projection=wcs)
        plt.imshow(data, vmin=0, vmax=5, origin='lower')
        # plt.scatter(pixels[0], pixels[1], marker='*', color='red')
        ax2.scatter(ra, dec, marker='o', facecolor='none', edgecolor='red', transform=ax2.get_transform('world'))
        ax2.add_patch(rect)
        plt.show()

    # Create the image cutout
    asdf_cutout = ASDFCutout(asdf_file_uri, center_coord, cutout_size, fill_value=0)
    
    # Check the cutout shape and position
    print(f'len of asdf_cutout.cutouts = {len(asdf_cutout.cutouts)}')
    cutout = asdf_cutout.cutouts
    gwcs_asdf = cutout[0].wcs
    
    
    scales = proj_plane_pixel_scales(gwcs_asdf) * 3600  # arcsec/pixel
    
    # x_pix = scales[0]
    # y_pix = scales[1]
    
    print('Cutout Data Shape:', cutout[0].shape)
    print('Cutout Original Position:', cutout[0].position_original)
    print('Cutout Position:', cutout[0].position_cutout)

    if display_cut == True:
        print('Displaying first image of three-filter cutouts of target')
        plt.figure()
        ax2 = plt.subplot(projection=gwcs_asdf)
        plt.imshow(cutout[0].data, vmin=0, vmax=5, origin='lower')
        plt.show()

    return cutout, scales

def Retrieve_file_matches(dir_uri, filter_match, All_files=True, in_plan=None, in_pass=None, in_segment=None):
    '''
    
    Retrieves a list of the files stored on AWS that match 
    the input dir_uri root directory, and input filter, plan, 
    pass, and segment. Also splits off separate arrays that 
    contain the visit-level info and sky-cell info associated 
    with each returned file. If All_files = True, returns all 
    files that match the input filter. Otherwise, an input 
    plan, pass, and segment 
    are also required.

    Parameters:
    -----------
    dir_uri: str
        the input directory to parse
    filter_match: str
        the filter to match against
    All_files: boolean
        Whether to return all files that match the input filter
        or impose filter on plan, pass, and segment as well
    in_plan: str
        The plan number. A specification in APT of the Pass Plans  
        comprising a Program. Should have format "01" for plan 1
    in_pass: str
        The pass number. The collection of activities generated from 
        each iteration of a Pass Plan. Multiple Passes may be generated 
        from the same Pass Plan to allow repetition or execute at 
        different orientations (via special requirements).
    in_segment: str
        The sequence of activities produced by each iteration of the 
        Observation Sequences defined in a Segment Plan. A Segment 
        may include multiple traversals of Mosaic Pattern(s), with 
        filter wheel moves occurring between Observations. The 
        Segment will be the unit of planning, while short term 
        scheduling will be done at the Visit level.
    
    Returns:
    --------
    visit_level_info_array: nd.array
        array of the viti-level combo information for each matched 
        program file.
    skycell_info_array: nd.array
        the skycell information for each matched program file
    prog_match_files: list
        the list of program files matched to the input filter, plan, 
        pass, and segment 
    '''
    file_list = fs.ls(dir_uri)
    print(f'Full length of file list available at {dir_uri} = ', len(file_list))
    
    print(f'Looking for files taken in filter {filter_match}')
    
    prog_match_files = []
    
    visit_level_info = []
    skycell_info = []
    filter_info = []
    suffix = []
    for i in range(len(file_list)):
    # for i in range(5): #v01001001002001
        suffix_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-1].rsplit('.')[0]
        visitlev_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-4]
        filter_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-2]
        skycell_inf = file_list[i].rsplit('/')[-1].rsplit('_')[-3]
        # print('visit-level-info = ', visitlev_inf)
        plan_current = visitlev_inf[1:3]
        pass_current = visitlev_inf[3:6]
        segment_current = visitlev_inf[6:9]
        # print(plan_current, pass_current, segment_current)
        
        
        # filter_info.append(filter_inf)
        if All_files == True:
            if filter_inf == filter_match and suffix_inf == 'coadd':

                visit_level_info.append(visitlev_inf)
                skycell_info.append(skycell_inf)
                prog_match_files.append(file_list[i])
        else:
            if filter_inf == filter_match and suffix_inf == 'coadd' and plan_current== in_plan and pass_current == in_pass and segment_current == in_segment:

                visit_level_info.append(visitlev_inf)
                skycell_info.append(skycell_inf)
                prog_match_files.append(file_list[i])

    print(f'Summary of files:')        
    if len(visit_level_info) > 0:
        len_fl = len(visit_level_info)
        print(f'{len_fl} files found that match input filter request!')
        # pprint(visit_level_info)
    else:
        print('No files found!')

    visit_level_info_array = np.array(visit_level_info)
    skycell_info_array = np.array(skycell_info)
        
    return visit_level_info_array, skycell_info_array, prog_match_files

def Print_file_info(visit_level_info_array, skycell_info_array):
    '''
    Given an input array of visit-level combinations and skycells, 
    print a summary of the visit-level and skycell information.  

    Prameters:
    ---------
    visit_level_info_array: nd.array
        an array of visit-level descriptors that summarize the 
        plan, pass, segment, observations and visit of each file
    
    skycell_info_array: nd.array
        an array of the skycell info associated with each 
        visit-level descriptor of the input program/filter

    Returns:
    --------
    None
        
    '''

    print('Number of unique visit-level combos :', len(np.unique(visit_level_info_array)))
    print('Number of unique skycell positions:', len(np.unique(skycell_info_array)))
    unique_visit_level_info = np.unique(visit_level_info_array)
    

    PASS_info = []
    SEG_info = []
    OBS_info = []
    VISIT_info = []
    SKYCELL_info = []
    
    
    for i in range(len(unique_visit_level_info)):
        VL_info = unique_visit_level_info[i]
        # these are hard coded to match the format of visit-level info of Roman WFI files
        # https://roman-docs.stsci.edu/data-handbook/data-levels-and-products
        pass_info = VL_info[3:6]
        seg_info = VL_info[6:9]
        obs_info = VL_info[9:12]
        visit_info = VL_info[12:]

        PASS_info.append(pass_info)
        SEG_info.append(seg_info)
        OBS_info.append(obs_info)
        VISIT_info.append(visit_info)
    
        visit_match = np.where(visit_level_info_array==VL_info)[0]
        skycell_match = skycell_info_array[visit_match]

        
    unique_PASS_info = np.unique(PASS_info)
    num_passes = len(unique_PASS_info)
    print(f'Found {num_passes} unique passes: ', unique_PASS_info)
    
    unique_SEF_info = np.unique(SEG_info)
    num_segs = len(unique_SEF_info)
    print(f'Found {num_segs} unique segments', unique_SEF_info)
    
    unique_OBS_info = np.unique(OBS_info)
    num_obs = len(unique_OBS_info)
    print(f'Found {num_obs} unique observations', unique_OBS_info)
    
    PASS_info_array = np.array(PASS_info)
    SEG_info_array = np.array(SEG_info)
    OBS_info_array = np.array(OBS_info)
    VISIT_info_array = np.array(VISIT_info)
    
    print('======================================')
    for pid in unique_PASS_info:

        id_match = np.where(PASS_info_array == pid)
        seg_match = SEG_info_array[id_match]
        obs_match = OBS_info_array[id_match]
        visit_match = VISIT_info_array[id_match]
    
        print(f'Pass {pid} has...')
        print(f'{len(np.unique(seg_match))} segment: {np.unique(seg_match)}')
        print(f'{len(np.unique(obs_match))} observations: {np.unique(obs_match)}')
        print(f'{len(np.unique(visit_match))} visits: {np.unique(visit_match)}')
        print('======================================')

### Check the available simulation images in the AWS S3 bucket for the SOC simulations:

- an example of the file naming structure: "r00342_p_v0100100100100_008m46x51y68_f087"

- the first part, "r00342," is the program

-"p" indicates that this is a prompt data product (not an officiel data release project)

- each file has a visit-level descriptor: "v01002001001001," which can be broken into components "v01|002|001|001|001"
    - This is a visit-level combination of co-added (dithered) exposures from execution plan 01, pass 002, segment 001, observation 001, visit 001. 

- pass 013 corresponds to the medium tier HLWAS (https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts)


- for each filter, this pass (013) should have plan 01, segment 001, and visits 001-018 (one for each tile of the HLWAS mosaic)


- for each filter, the "observation" descriptor varies: f106: 001; f129: 002; f158: 003


- Need to indentify all files that correspond to plan 01, pass 013, and segment 001


- the second-to-last part of the file naming structure is the skycell. The skycell information "008m46x51y68" contains the center of the projection region (008m46 = RA,DEC = 08, -46) and the center of the sky cell within the proection region in pixel coords (x51y68 = x,y = 51,68)


In [ ]:
'''
find out how many unique visit-level combinations for each filter to help get an idea of what "observations" this dataset includes
'''
# file_source = 'openuniverse2024'
file_source = 'soc_simulations'

# list all of the available simulated dataproducts
# we need filters YJH
in_filter = 'f106' #primary filter
f1 = 'f129' #secondary filter
f2 = 'f158' #tertiary filter
bands = [in_filter, f1, f2]
src_mag_limit=23.5 #magntiude limit is somewhat arbitrary, chosen to match the magntiude limit of mos galaxies in the GZ: Hubble project

in_plan_ = '01'
in_pass_ = '013'
in_segment_ = '001'
# in_observation = '002'
# in_visit = '001'

fs = s3fs.S3FileSystem(anon=True) #connects us to AWS S3 bucket where the data is stored; anon=True means we only want access to public data files


if file_source == 'soc_simulations':
    
    infilter_visit_level_info_array, infilter_skycell_info_array, infilter_prog_match_files = Retrieve_file_matches('s3://stpubdata/roman/nexus/soc_simulations/r00342/l3/', in_filter, All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)
    Print_file_info(infilter_visit_level_info_array, infilter_skycell_info_array)
        
    f1_visit_level_info_array, f1_skycell_info_array, f1_prog_match_files = Retrieve_file_matches('s3://stpubdata/roman/nexus/soc_simulations/r00342/l3/', f1, All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)
    Print_file_info(f1_visit_level_info_array, f1_skycell_info_array)
    
    f2_visit_level_info_array, f2_skycell_info_array, f2_prog_match_files = Retrieve_file_matches('s3://stpubdata/roman/nexus/soc_simulations/r00342/l3/', f2, All_files=False, in_plan=in_plan_, in_pass=in_pass_, in_segment=in_segment_)
    Print_file_info(f2_visit_level_info_array, f2_skycell_info_array)

    

# Start from the visit-level combinations returned for the f106 filter.
- for every program file that matches the in_filter and pass_number (which is a observation + visit + skycell combination), identify the file name for the match_filters that overlap with the skycell
- the matched files in each filter may contain up to 4 visits (single pointings) that I'm guessing overlap at the given skycell. The visit numbers should be adjacent to each other in the mosaic (e.g., 1,2,11,12 or 2,3,10,11 becuase simulated moasic proceeds in an S-pattern - see https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts )
- ##### what is the best way to select the "best" visit for a given skycell (for skycells where multiple pointings (visits) overlap??)
    - could take the first one in the returned list but it may not cover the galaxy completely and we'd have to figure out which one is the best?
    - could create all returned images, but then still need to decide how to deduplicate later on...
    - also, some visits may have more coverage of the target than others...how to figure out which visit has the most coverage of the target?

In [ ]:
# organize the visit-level and sky-cell info for each non-primary filter
match_filters = [f1, f2]
match_filters_info = {f1:{'visit_level_info': f1_visit_level_info_array, 
                          'skycell_info_array':f1_skycell_info_array, 
                          'prog_match_files':f1_prog_match_files}, 
                      f2:{'visit_level_info': f2_visit_level_info_array, 
                          'skycell_info_array':f2_skycell_info_array, 
                          'prog_match_files':f2_prog_match_files}}

# the root directory of the program
dir_uri = 's3://stpubdata/roman/nexus/soc_simulations/r00342/'




# id_str	fits_loc_F115W	fits_loc_F150W	fits_loc_F277W	fits_loc_F444W	field	RA_MODEL	DEC_MODEL	AREA	MAG_MODEL_F444W	AXRATIO

print(in_filter, f1, f2)
catalog_df = pd.DataFrame(columns=['id_str',f'asdf_loc_{in_filter}',f'asdf_loc_{f1}',f'asdf_loc_{f2}', 'RA', 'DEC', f'KronABmag_{in_filter}'])

print(catalog_df)

# catalog_df[f'fits_loc_{f1}']= ['a test string']
# print(catalog_df[f'fits_loc_{f1}'])
# bands

In [ ]:
# for i in range(len(infilter_prog_match_files)):
source_number = []
in_filter_asdf_file_loc = []
f1_asdf_file_loc = []
f2_asdf_file_loc = []
source_RA = []
source_DEC = []
source_in_filter_KronABmag = []
for i in range(3): 
    # =============================================================================================
    # define primary filter image and catalog URIs for the current visit- and skycell-level descriptor
    file_name_coadd = 's3://'+infilter_prog_match_files[i]
    file_name_cat = dir_uri+'l4/l3_source_catalog/'+file_name_coadd.rsplit('/')[-1].rstrip('_coadd.asdf')+'_cat.parquet'

    print(f'INPUT FILE NAME: {file_name_coadd}')
    print(f'USING L4 CATALOG {file_name_cat} \n')
    
    # =============================================================================================
    # create empty dictionary to store the full URIs that cover the same visit- and skycell-level descriptor as the primary filter 
    MATCH_FILTER_IMAGES = {f1:{'prog_match_files':None}, 
                           f2:{'prog_match_files':None}}

    # for the secondary and tertiary filters, identify where their skycell info matches the current primary filter skycell info
    for f in match_filters:
        match_i = np.where(match_filters_info[f]['skycell_info_array']==infilter_skycell_info_array[i])[0]
        
        print(f'    The following images were indentified for filter {f}:')
        pprint(np.array(match_filters_info[f]['prog_match_files'])[match_i])

        len_match = len(np.array(match_filters_info[f]['prog_match_files'])[match_i])
        if len_match>0:
            MATCH_FILTER_IMAGES[f]['prog_match_files'] = np.array(match_filters_info[f]['prog_match_files'])[match_i]
    
        # report how many pointings (visits) of the secondary or tertiary filter overlap with the current visit- and skycell-level descriptor of the primary filter 
        print(f"    {len_match} pointings in filter {f} match the input skycell \n")
        

        # Once at the tertiary filter, check that files were found
        if f == match_filters[-1] and isinstance(MATCH_FILTER_IMAGES[f]['prog_match_files'], np.ndarray) and isinstance(MATCH_FILTER_IMAGES[match_filters[0]]['prog_match_files'], np.ndarray):
            print('    Found skycell file matches for all filters! Can create three color images! \n')
            pprint(MATCH_FILTER_IMAGES)
            print('\n')
            if len(MATCH_FILTER_IMAGES[f]['prog_match_files']) > 1:
                print('    Creating three filter images for the first pointing of input filter match list \n')
                image_file_f1 = 's3://'+MATCH_FILTER_IMAGES[f1]['prog_match_files'][0]
                image_file_f2 = 's3://'+MATCH_FILTER_IMAGES[f2]['prog_match_files'][0]
                # print(file_name_cat)
                # print(file_name_coadd)
                # print(image_file_f1) 
                # print(image_file_f2)
                
                # open the image and extract data array and metadata
                data_, allwcs_, pixel_scale_ = open_image_asdf(file_name_coadd)
                wcs_ = allwcs_[0]
                gwcs_ = allwcs_[1]
                
                # =============================================================================================
                # Get info on sky footprint of image
                # minRA_,minDEC_,maxRA_,maxDEC_ = get_footprint_gwcs(gwcs_)
            
            
                # #  ===================================================================================
                # read in associated catalog file:
                # Read the file metadata schema
                parquet_file = pq.read_table(file_name_cat, filesystem=fs)
                columns = parquet_file.schema.names
                
                cat_table = parquet_file.to_pandas()
                print(cat_table.columns)
                # extract RA,DEC positions for all sources in the current catalog
                srcRA_, srcDEC_, kron_abmag = np.array(cat_table['ra'].values), np.array(cat_table['dec'].values), np.array(cat_table['kron_abmag'].values)

                # identify which catalog sources are extended and have kron_abmag < 23.5
                ext_i = np.where((cat_table['is_extended']) & (kron_abmag<src_mag_limit))[0] 

                # extract RA,DEC positions for sources in the current catalog that meet the selection criteria
                srcRA_, srcDEC_, kron_abmag = srcRA_[ext_i], srcDEC_[ext_i], kron_abmag[ext_i]
                
                print(f'number of extended sources in this skycell w/ Kron AB magnitude < {src_mag_limit} = {len(ext_i)} \n')
                
                print('Displaying image with identified sources overlaid')
                plt.figure(figsize=(10, 8))
                ax2 = plt.subplot(projection=wcs_)
                plt.imshow(data_, vmin=0, vmax=5, origin='lower')
                # plt.scatter(pixels[0], pixels[1], marker='*', color='red')
                ax2.scatter(srcRA_, srcDEC_, marker='o', facecolor='none', edgecolor='red', transform=ax2.get_transform('world'))
                plt.show()

                # # create cutouts of each object in query results...
                cutout_size_phys = 5 #arcsec
                cutout_size_ = [int(cutout_size_phys/pixel_scale_[0]),int(cutout_size_phys/pixel_scale_[1])] #arcsec
                print('cutout size = ', cutout_size_)
                for l in range(len(srcRA_)):

                    ra_, dec_ = srcRA_[l], srcDEC_[l]

                    im_cut, scales_ = create_image_cutout_test([file_name_coadd, image_file_f1, image_file_f2], cutout_size_, [srcRA_[l], srcDEC_[l]], display_full=False, full_data = [data_,wcs_])
                    
                    cutouts_ = im_cut
                    
                    in_filter_asdf_file_loc.append(file_name_coadd)
                    f1_asdf_file_loc.append(image_file_f1)
                    f2_asdf_file_loc.append(image_file_f2)
                    source_RA.append(ra_)
                    source_DEC.append(dec_)
                    source_in_filter_KronABmag.append(kron_abmag[l])

                    
     
                    for c in range(len(cutouts_)):
                        plt.figure(figsize=(10, 8))
                        ax2 = plt.subplot(projection=cutouts_[c].wcs)
                        plt.imshow(cutouts_[c].data, origin='lower', norm=LogNorm(vmin=0.01, vmax=np.nanmax(cutouts_[c].data)))
                        # plt.scatter(pixels[0], pixels[1], marker='*', color='red')
                        ax2.scatter(ra_, dec_, marker='o', facecolor='none', edgecolor='red', transform=ax2.get_transform('world'))
                        plt.show()
    print('==============================================================================')

In [ ]:
# print(in_filter_asdf_file_loc)
# src_i = 13
# cutout_ = Cutout2D(data_, center_coord_, (cutout_size_[0],cutout_size_[1]), wcs=wcs_)
# im_cut, scales_ = create_image_cutout_test([in_filter_asdf_file_loc[src_i], f1_asdf_file_loc[src_i], f2_asdf_file_loc[src_i]], cutout_size_, [source_RA[src_i], source_DEC[src_i]], display_full=False)


In [ ]:
for src_i in range(len(in_filter_asdf_file_loc)):
    im_cut, scales_ = create_image_cutout_test([in_filter_asdf_file_loc[src_i], f1_asdf_file_loc[src_i], f2_asdf_file_loc[src_i]], cutout_size_, [source_RA[src_i], source_DEC[src_i]], display_cut=False, display_full=False)
    cutouts_ = im_cut
    # If any of the bands is just 0's, don't run code
    if cutouts_[0].data.max()== 0 or cutouts_[1].data.max() == 0 or cutouts_[2].data.max() == 0:
        print('Wrong')
    f1_, f2_, f3_ = cutouts_[0].data, cutouts_[1].data, cutouts_[2].data
    
    # print(f1_, f2_, f3_)
    # Step 2: make arsinh and mtf rgbs
    try:
        rgb_arcsinh = make_arcsinh_rgb(f1_, f2_, f3_, weights = None).astype(np.uint8)
        
        rgb_mtf = make_mtf_rgb(f1_, f2_, f3_, weights = None).astype(np.uint8)
    except:
        print('Failed to make image!')
    
    
    
    # plt.figure()
    # plt.imshow(rgb_arcsinh)
    # plt.show()
    
    
    # plt.figure()
    # plt.imshow(rgb_mtf)
    # plt.show()
    
    # Step 3: desaturate
    rgb_arcsinh_desaturated = desaturate(rgb_arcsinh)
    rgb_mtf_desaturated = desaturate(rgb_mtf)
    
    # Step 4: visualise
    fig, axes = plt.subplots(ncols=4)
    axes[0].imshow(rgb_arcsinh)
    axes[0].axis('off')
    axes[0].set_title('arcsinh', fontsize = 12)
    
    axes[1].imshow(rgb_arcsinh_desaturated)
    axes[1].axis('off')
    axes[1].set_title('arcsinh desaturated', fontsize = 10)
    
    axes[2].imshow(rgb_mtf)
    axes[2].axis('off')
    axes[2].set_title('mtf', fontsize = 12)
    
    axes[3].imshow(rgb_mtf_desaturated)
    axes[3].axis('off')
    axes[3].set_title('mtf desaturated', fontsize = 10)
    
    plt.tight_layout()
    plt.show()
    # for c in range(len(cutouts_)):
    #     plt.figure(figsize=(10, 8))
    #     ax2 = plt.subplot(projection=cutouts_[c].wcs)
    #     plt.imshow(cutouts_[c].data, origin='lower', norm=LogNorm(vmin=0.01, vmax=np.nanmax(cutouts_[c].data)))
    #     # plt.scatter(pixels[0], pixels[1], marker='*', color='red')
    #     # ax2.scatter(ra_, dec_, marker='o', facecolor='none', edgecolor='red', transform=ax2.get_transform('world'))
    #     plt.show()

In [ ]:
def MTF_on_normalised_data(x, m):
    """
    Compute the Midtones Transfer Function (MTF) for given x and m.
    This is basically a way to attempt to automatically set the "curves" tool from e.g. photoshop, 
    where m sets the curve and MTF is the application of the curve."""
    # x = np.clip(x, 0, 1)
    assert x.max() <= 1
    assert x.min() >= 0
    y = np.zeros_like(x)
    mask0 = (x == 0)
    maskm = (x == m)
    mask1 = (x == 1)
    mask_else = ~(mask0 | maskm | mask1)
    # these remain fixed
    y[mask0] = 0
    y[maskm] = 0.5
    y[mask1] = 1
    # ..and everything else gets curved
    x_else = x[mask_else]

    # shape of the curve
    numerator = (m - 1) * x_else
    denominator = (2 * m - 1) * x_else - m
    # apply the curve
    y[mask_else] = numerator / denominator
    return y # curved values
    
def find_m_for_mean(normalized_data, desired_mean, central_crop_size=100, central_crop_fraction=None, quick_mask=False):  # 0.1 arcsec per pixel for MER tiles

    
    width = normalized_data.shape[1]
    
    if quick_mask:
        normalized_data_selected = make_quick_mask(normalized_data)
    else:
        if central_crop_fraction is not None:
            central_crop_size = int(central_crop_fraction * width)

        central_crop_low_edge = width//2 - central_crop_size//2
        central_crop_high_edge = width//2 + central_crop_size//2
        normalized_data_selected = normalized_data[central_crop_low_edge:central_crop_high_edge, central_crop_low_edge:central_crop_high_edge]

    # print(normalized_data_selected.sum(), normalized_data_selected.mean())

    x = np.mean(normalized_data_selected)
    alpha = desired_mean
    return (x-alpha*x)/(x-2*alpha*x+alpha)

def apply_MTF(image_data, desired_mean_normalized=0.2, central_crop_size=100, central_crop_fraction=None, quick_mask=False, percentile=99.5):  
    image_data = np.nan_to_num(image_data, nan=0.0, posinf=0.0, neginf=0.0)
    
    if percentile is not None:
            image_data = np.clip(image_data, image_data.min(), np.percentile(image_data, percentile))
    
    # Normalize the image data to [0, 1]
    normalized_data = image_data.astype(np.float64)
    min_val = np.min(normalized_data)
    max_val = np.max(normalized_data)
    normalized_data = (normalized_data - min_val) / (max_val - min_val)
    # Desired mean value at 1/5 of the maximum pixel value
    # Raised from 1/8 default from Tian
    # Find the appropriate midtones balance parameter m
    m = find_m_for_mean(normalized_data, desired_mean_normalized, central_crop_size, central_crop_fraction, quick_mask)
    # Apply the MTF to the image
    transformed_data = MTF_on_normalised_data(normalized_data, m)
    # Scale transformed data to [0, 255] for JPEG output
    transformed_image_data = (transformed_data * 255).astype(np.uint8)
    return transformed_image_data
    
def make_mtf_rgb(f1, f2, f3, weights = {}, means=[0.27, 0.25, 0.23, 0.25]):  # last not used
    
    mtf_args = {'central_crop_fraction': 1., 'percentile': 99.85}
    f1_mtf = apply_MTF(f1, **mtf_args, desired_mean_normalized=means[0])
    f2_mtf = apply_MTF(f2, **mtf_args, desired_mean_normalized=means[1])
    f3_mtf = apply_MTF(f3, **mtf_args, desired_mean_normalized=means[2])
    # print(f1_mtf, f2_mtf, f3_mtf)
    #weights = jeyhan_color_weights
    #weights = minimal_color_weights
    #weights = klm_color_weights
    
    # if not weights: #if you don't specify weights, use original default weights sent by Jeyhan
    #     weights = dict(
    #         f115w = (0, 0, 1), 
    #         f150w = (0, 0.3, 0.7),
    #         f277w = (0.7, 0.3, 0),
    #         f444w = (1, 0, 0)
    #     )

    red = f1_mtf
    # f115_mtf*weights['f115w'][0] + f150_mtf*weights['f150w'][0] + f277_mtf*weights['f277w'][0] + f444_mtf*weights['f444w'][0]
    green = f2_mtf
    # f115_mtf*weights['f115w'][1] + f150_mtf*weights['f150w'][1] + f277_mtf*weights['f277w'][1] + f444_mtf*weights['f444w'][1]
    blue = f3_mtf
    # f115_mtf*weights['f115w'][2] + f150_mtf*weights['f150w'][2] + f277_mtf*weights['f277w'][2] + f444_mtf*weights['f444w'][2]
    
    return np.stack([red, green, blue], axis=2)

In [ ]:

def adjust_dynamic_range(flux, q=100, clip=99.85):
    '''
    from https://github.com/mwalmsley/bulk-euclid-cutouts/blob/main/bulk_euclid/utils/morphology_utils_ou_mer.py
    '''
    im = np.arcsinh(flux * q)
    if clip < 100:
        im = np.clip(im, 0, np.percentile(im, clip))
    return  im

def to_uint8(im, clip_below_zero=True):
    '''
    from https://github.com/mwalmsley/bulk-euclid-cutouts/blob/main/bulk_euclid/utils/morphology_utils_ou_mer.py
    '''
    if clip_below_zero:
        im = np.clip(im, 0, None)
    im_min = np.nanmin(im)
    im_max = np.nanmax(im)
    assert im_min < im_max, f'Image min {im_min} >= max {im_max}, likely all data is zero or NAN i.e. missing'
    im = (im + im_min) / (im_min + im_max)
    return (255 * im).astype(np.uint8)
    
def make_vis_only_cutout(vis_cutout, q=100, clip=99.85):
    '''
    from https://github.com/mwalmsley/bulk-euclid-cutouts/blob/main/bulk_euclid/utils/morphology_utils_ou_mer.py
    '''
    vis_flux_adjusted = adjust_dynamic_range(vis_cutout, q=q, clip=clip)
    vis_uint8 = to_uint8(vis_flux_adjusted)
    return vis_uint8

def make_arcsinh_rgb(f1, f2, f3, q=[1, 1, 1], weights = {}):
    f1_arcsinh = make_vis_only_cutout(f1, q=q[0], clip=99.85)
    f2_arcsinh = make_vis_only_cutout(f2, q=q[1], clip=99.85)
    f3_arcsinh = make_vis_only_cutout(f3, q=q[2], clip=99.85)
    # print(f1_arcsinh, f2_arcsinh, f3_arcsinh)
    # f444_arcsinh = make_vis_only_cutout(f444, q=q[3], clip=99.85)
    # print('f115_arcsinh', f115_arcsinh)
    # print('f150_arcsinh', f150_arcsinh)
    # print('f277_arcsinh', f277_arcsinh)
    #weights = jeyhan_color_weights
    #weights = minimal_color_weights
    #weights = klm_color_weights
    
    # if not weights: #if you don't specify weights, use original default weights sent by Jeyhan
    #     weights = dict(
    #         f115w = (0, 0, 1), 
    #         f150w = (0, 0.3, 0.7),
    #         f277w = (0.7, 0.3, 0),
    #         f444w = (1, 0, 0)
    #     )
        

    red = f1_arcsinh
    # *weights['f115w'][0] + f150_arcsinh*weights['f150w'][0] + f277_arcsinh*weights['f277w'][0] + f444_arcsinh*weights['f444w'][0]
    green = f2_arcsinh
    # *weights['f115w'][1] + f150_arcsinh*weights['f150w'][1] + f277_arcsinh*weights['f277w'][1] + f444_arcsinh*weights['f444w'][1]
    blue = f3_arcsinh
    # *weights['f115w'][2] + f150_arcsinh*weights['f150w'][2] + f277_arcsinh*weights['f277w'][2] + f444_arcsinh*weights['f444w'][2]
    # print('[red, green, blue]', [red, green, blue])
    return np.stack([red, green, blue], axis=2)

def desaturate(rgb):
    '''
    Adapted from https://github.com/zooniverse/decals/blob/master/decals/a_download_decals/get_images/image_utils.py#L77
    '''
    # optionally desaturate pixels that are dominated by a single
    # colour to avoid colourful speckled sky
    
    rgb = rgb.astype(np.float32) / 255.0 #Rescale so that it is between 0 and 1.
    h, w = rgb.shape[0:2]

    # reshape rgb from (h, w, 3) to (3, h, w)
    RGBim = np.array([rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]])
    a = RGBim.mean(axis=0)  # a is mean pixel value across all bands, (h, w) shape
    # putmask: given array and mask, set all mask=True values of array to new value
    np.putmask(a, a == 0.0, 1.0)  # set pixels with 0 mean value to mean of 1. Inplace? Avoids dividing by zero.
    acube = np.resize(a, (3, h, w))  # copy mean value array (h,w) into 3 bands (3, h, w). So acube has now the same value in each band
    bcube = (RGBim / acube) / 2.5  # bcube: divide image by mean-across-bands pixel value, and again by 2.5 (why?). I think this step normalizes band values compared to the mean, then scaling it down. 2.5 is probably the threshold for what it means to be "saturated". E.g. if one band has 2.5 times the average flux compared to the mean of all bands.
    mask = np.array(bcube)  # isn't bcube already an array?
    wt = np.max(mask, axis=0)  # maximum per pixel across bands of mean-band-normalised rescaled image
    # i.e largest relative deviation from mean
    np.putmask(wt, wt > 1.0, 1.0)  # clip largest allowed relative deviation to one (inplace?)
    wt = 1 - wt  # invert relative deviations. Now a value of 0 means that one band clearly donimates.
    wt = np.sin(wt*np.pi/2.0)  # non-linear rescaling of relative deviations. If wt was 0, now the output becomes 0. If it was 1, also stays 1. Smooth mapping in between. So wt is a w x h array with values between 0 (dominated by one band) and 1 (not dominated).
    temp = RGBim * wt + a*(1-wt) + a*(1-wt)**2 * RGBim  # multiply by weights in complicated fashion
    # AI interpretation of this equation: new color = original color + gray version (mean) + contrast restoration
    # RGBim * wt: downweight the saturated pixels (should have wt=0)
    # a*(1-wt): shifts saturated pixels back towards the average pixel value across all filters (i.e., downweight, but then add the downweighted value to something close to the mean
    # a*(1-wt)**2 * RGBim: add back some information about contrast between colors (contained in RGBim) for pixels that are close to/are saturated, with the amount of contrast determined relative to how close to saturated the pixel color is

    rgb = np.zeros((h, w, 3), np.float32)  # reset rgb to be blank
    for idx, im in enumerate((temp[0, :, :], temp[1, :, :], temp[2, :, :])):  # fill rgb with weight-rescaled rgb
        rgb[:, :, idx] = im

    rgb = np.clip(rgb, 0., 1.) # clip at 0 and 1 
    rgb = (rgb * 255).astype(np.uint8) # Rescale back to 0 - 255 scale
        
    return rgb

In [ ]:
# If any of the bands is just 0's, don't run code
if cutouts_[0].data.max()== 0 or cutouts_[1].data.max() == 0 or cutouts_[2].data.max() == 0:
    print('Wrong')
f1_, f2_, f3_ = cutouts_[0].data, cutouts_[1].data, cutouts_[2].data

# print(f1_, f2_, f3_)
# Step 2: make arsinh and mtf rgbs
rgb_arcsinh = make_arcsinh_rgb(f1_, f2_, f3_, weights = None).astype(np.uint8)

rgb_mtf = make_mtf_rgb(f1_, f2_, f3_, weights = None).astype(np.uint8)




plt.figure()
plt.imshow(rgb_arcsinh)
plt.show()


plt.figure()
plt.imshow(rgb_mtf)
plt.show()

# Step 3: desaturate
rgb_arcsinh_desaturated = desaturate(rgb_arcsinh)
rgb_mtf_desaturated = desaturate(rgb_mtf)

# Step 4: visualise
fig, axes = plt.subplots(ncols=4)
axes[0].imshow(rgb_arcsinh)
axes[0].axis('off')
axes[0].set_title('arcsinh', fontsize = 12)

axes[1].imshow(rgb_arcsinh_desaturated)
axes[1].axis('off')
axes[1].set_title('arcsinh desaturated', fontsize = 10)

axes[2].imshow(rgb_mtf)
axes[2].axis('off')
axes[2].set_title('mtf', fontsize = 12)

axes[3].imshow(rgb_mtf_desaturated)
axes[3].axis('off')
axes[3].set_title('mtf desaturated', fontsize = 10)

plt.tight_layout()
plt.show()